# Daily Practice — 2026-10-19: Statistics — Is That Slice Gap Real, or Just Noise?

**Dataset:** [Palmer Archipelago Penguins](https://allisonhorst.github.io/palmerpenguins/) (344 real penguins measured by researchers at Palmer Station, Antarctica — bill length/depth, flipper length, body mass, species, island, sex), fetched live from seaborn's public dataset repo on GitHub. This is genuine field-collected biological data, not a synthetic set.

**The scenario:** You're the QA engineer signing off on a classifier that predicts penguin `species` from body measurements before it ships. Overall test accuracy looks great. But your team's slice-based evaluation dashboard (the kind you'd build for any production ML system) shows the model's accuracy is *not identical* across the three islands the penguins were measured on — Biscoe, Dream, and Torgersen. Someone on the team wants to file a "model is biased against Torgersen penguins" bug. Your job is to determine, with statistics rather than a gut feeling, whether that gap is a real, reproducible effect or just sampling noise — and to do it *correctly* when there are multiple island pairs to compare, so you don't cry wolf (or miss a real issue) because of multiple-comparisons inflation.

**What you should produce:**
1. A trained baseline classifier (`species` from the four numeric measurements) with an overall held-out test accuracy.
2. A **bootstrap 95% confidence interval** for that overall accuracy (resampling the test set, not refitting the model).
3. A **per-island accuracy table** on the test set (slice-based evaluation).
4. For every pair of islands, a **permutation test** p-value for "is the accuracy gap between these two islands bigger than chance?".
5. The same set of p-values after **Holm–Bonferroni correction** for multiple comparisons, and a clear statement of which pairwise gaps (if any) survive correction.
6. A short written verdict (2–3 sentences, in a markdown cell): do you file the bias bug, or close it as noise? Justify it using your numbers, not vibes.

Use `random_state=0` everywhere you sample, split, or permute, so your results are reproducible run to run — reproducibility is itself part of the QA bar here.

## Setup and data loading

Nothing to fill in here — just imports and getting a clean dataframe to work with.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score

RNG_SEED = 0

url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"
penguins = pd.read_csv(url)

FEATURES = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
penguins = penguins.dropna(subset=FEATURES + ["species", "island"]).reset_index(drop=True)

print(penguins.shape)
penguins.head()

In [ ]:
X = penguins[FEATURES].to_numpy()
y = penguins["species"].to_numpy()
islands = penguins["island"].to_numpy()

X_train, X_test, y_train, y_test, island_train, island_test = train_test_split(
    X, y, islands,
    test_size=0.3,
    stratify=y,
    random_state=RNG_SEED,
)

clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=RNG_SEED))
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
overall_acc = accuracy_score(y_test, y_pred)
print(f"Overall test accuracy: {overall_acc:.4f}  (n_test = {len(y_test)})")

## Your turn — fill in the functions below

Implement each TODO. Keep every function's `random_state` plumbed through to `np.random.default_rng(random_state)` (or equivalent) so re-running with the same seed gives identical numbers.

In [ ]:
def bootstrap_ci(y_true, y_pred, metric_fn=accuracy_score, n_boot=2000, ci=0.95, random_state=0):
    '''
    Bootstrap confidence interval for a metric computed on (y_true, y_pred).

    Resample the *indices* of the test set with replacement n_boot times,
    recompute metric_fn on each resample, and return
    (point_estimate, lower_bound, upper_bound) using the percentile method.

    Do NOT refit any model here -- y_true/y_pred are already fixed predictions;
    bootstrapping just quantifies sampling uncertainty in the metric itself.
    '''
    # TODO: implement
    raise NotImplementedError


def slice_accuracy(y_true, y_pred, slice_labels):
    '''
    Return a dict {slice_value: accuracy} of accuracy computed separately
    within each distinct value of slice_labels (e.g. each island).
    Also include the slice's sample count alongside accuracy, e.g.
    {slice_value: (accuracy, n)}.
    '''
    # TODO: implement
    raise NotImplementedError


def permutation_test_pair(y_true, y_pred, slice_labels, group_a, group_b, n_perm=2000, random_state=0):
    '''
    Permutation test for whether the accuracy gap between group_a and group_b
    (two values found in slice_labels) is bigger than you'd expect by chance.

    Steps:
      1. Restrict to rows where slice_labels is group_a or group_b.
      2. Compute the observed accuracy gap: acc(group_a) - acc(group_b).
      3. n_perm times: randomly reassign the group_a/group_b labels among
         those same rows (keeping y_true/y_pred fixed), recompute the gap,
         building a null distribution of gaps under "island label doesn't matter".
      4. Return (observed_gap, two_sided_p_value), where the p-value is the
         fraction of permuted |gap| at least as extreme as the observed |gap|
         (use the (count + 1) / (n_perm + 1) smoothing to avoid a p-value of 0).
    '''
    # TODO: implement
    raise NotImplementedError


def holm_bonferroni(p_values, alpha=0.05):
    '''
    Holm-Bonferroni step-down correction for a list of p-values.

    Return a list of booleans (same order as input) that is True where the
    null hypothesis is rejected at family-wise level alpha, i.e. the result
    is still "significant" after correcting for doing len(p_values) tests.
    '''
    # TODO: implement
    raise NotImplementedError

In [ ]:
# TODO: use the functions above to produce:
#   1. overall_point, overall_lo, overall_hi = bootstrap_ci(y_test, y_pred)
#   2. a per-island accuracy table (as a pandas DataFrame, sorted by accuracy)
#   3. observed gap + raw p-value for every one of the 3 island pairs
#      (Biscoe vs Dream, Biscoe vs Torgersen, Dream vs Torgersen)
#   4. the Holm-Bonferroni-adjusted significance verdict for those 3 p-values
#
# Print a results table that makes it obvious, at a glance, which pairwise
# gaps are "significant" before correction and which survive after correction.


**Your verdict** (2-3 sentences): based on your numbers above, do you file the "model is biased against Torgersen" bug, or close it as sampling noise? Would your answer change if you *hadn't* corrected for multiple comparisons?

_(write your answer here)_

---

## Solution

*(scroll down when you're ready — try it yourself first)*

<details>
<summary>Click to reveal solution</summary>

```python
def bootstrap_ci(y_true, y_pred, metric_fn=accuracy_score, n_boot=2000, ci=0.95, random_state=0):
    rng = np.random.default_rng(random_state)
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    n = len(y_true)
    point = metric_fn(y_true, y_pred)

    boot_stats = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n, size=n)
        boot_stats[b] = metric_fn(y_true[idx], y_pred[idx])

    alpha = 1 - ci
    lo, hi = np.percentile(boot_stats, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return point, lo, hi


def slice_accuracy(y_true, y_pred, slice_labels):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    slice_labels = np.asarray(slice_labels)
    out = {}
    for value in np.unique(slice_labels):
        mask = slice_labels == value
        out[value] = (accuracy_score(y_true[mask], y_pred[mask]), int(mask.sum()))
    return out


def permutation_test_pair(y_true, y_pred, slice_labels, group_a, group_b, n_perm=2000, random_state=0):
    rng = np.random.default_rng(random_state)
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    slice_labels = np.asarray(slice_labels)

    mask = np.isin(slice_labels, [group_a, group_b])
    yt, yp, labels = y_true[mask], y_pred[mask], slice_labels[mask].copy()
    correct = (yt == yp).astype(float)

    def gap(lbls):
        return correct[lbls == group_a].mean() - correct[lbls == group_b].mean()

    observed = gap(labels)

    perm_gaps = np.empty(n_perm)
    for i in range(n_perm):
        perm_labels = rng.permutation(labels)
        perm_gaps[i] = gap(perm_labels)

    extreme = np.sum(np.abs(perm_gaps) >= np.abs(observed))
    p_value = (extreme + 1) / (n_perm + 1)
    return observed, p_value


def holm_bonferroni(p_values, alpha=0.05):
    p_values = np.asarray(p_values, dtype=float)
    order = np.argsort(p_values)
    m = len(p_values)
    reject_sorted = np.zeros(m, dtype=bool)

    for rank, idx in enumerate(order):
        threshold = alpha / (m - rank)
        if p_values[idx] <= threshold:
            reject_sorted[rank] = True
        else:
            break  # Holm's step-down: stop at first failure to reject

    reject = np.zeros(m, dtype=bool)
    reject[order] = reject_sorted
    return reject.tolist()
```

**Running it:**

```python
overall_point, overall_lo, overall_hi = bootstrap_ci(y_test, y_pred, n_boot=2000, random_state=RNG_SEED)
print(f"Overall accuracy: {overall_point:.3f}  95% bootstrap CI [{overall_lo:.3f}, {overall_hi:.3f}]")

per_island = slice_accuracy(y_test, y_pred, island_test)
island_df = pd.DataFrame(
    [(island, acc, n) for island, (acc, n) in per_island.items()],
    columns=["island", "accuracy", "n"],
).sort_values("accuracy", ascending=False).reset_index(drop=True)
print(island_df)

pairs = [("Biscoe", "Dream"), ("Biscoe", "Torgersen"), ("Dream", "Torgersen")]
results = []
for a, b in pairs:
    gap, p = permutation_test_pair(y_test, y_pred, island_test, a, b, n_perm=2000, random_state=RNG_SEED)
    results.append({"pair": f"{a} vs {b}", "observed_gap": gap, "raw_p": p})

results_df = pd.DataFrame(results)
results_df["holm_significant"] = holm_bonferroni(results_df["raw_p"].to_list(), alpha=0.05)
results_df["raw_significant"] = results_df["raw_p"] < 0.05
print(results_df)
```

**Actual output with `random_state=0`, `test_size=0.3`, `n_boot=n_perm=2000`:**

```
Overall accuracy: 0.990  95% bootstrap CI [0.971, 1.000]

      island  accuracy   n
      Biscoe    1.0000  49
       Dream    1.0000  38
   Torgersen    0.9375  16

               pair  observed_gap   raw_p  holm_significant  raw_significant
    Biscoe vs Dream        0.0000  1.0000             False            False
Biscoe vs Torgersen        0.0625  0.2444             False            False
 Dream vs Torgersen        0.0625  0.2889             False            False
```

**Interpretation:** the dashboard number that would trigger a bug report is the Torgersen row — 93.75% vs. a perfect 100% elsewhere, a ~6-point gap on just 16 test penguins. But the permutation test already puts that gap's raw p-value around 0.24–0.29: *before any multiple-comparisons correction at all*, a 6-point gap on 16 samples is comfortably inside what you'd expect from reshuffling island labels at random. Holm-Bonferroni on these three p-values changes nothing here because none of them was close to significant to begin with — but that's the point of running the permutation test instead of eyeballing the table: a gap that *looks* dramatic (100% → 93.75%) can be completely unremarkable once you account for the tiny slice size, and the correction step is what protects you on days when a seed or feature change *does* produce one marginal p-value among several slices, so you don't chase it as a real regression.

**Verdict:** close the "model is biased against Torgersen" report as sampling noise, not a bug — the per-island accuracy table alone is misleading at this sample size, and the permutation test shows the gap is well within chance. Had you skipped the significance test and multiple-comparisons correction and just reacted to the raw accuracy table, you'd have escalated a false alarm. (Re-run with a different `random_state` in the train/test split and you may occasionally see a larger gap or a smaller raw p-value for one pair — that's exactly when the Holm correction earns its keep, since testing three pairs instead of one inflates your chance of at least one spuriously small p-value to roughly 1-(0.95)³ ≈ 14% if you don't correct for it.)

</details>